# Lecture 10 · Notebook 1 — CLIP: zero-shot classification and image–text retrieval
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture10/lecture10_1_clip_zero_shot.ipynb)

**Companion to [Part 2 of the video lecture](https://nlp-course-cuhksz.github.io/lecture10.html#p2-s13)** (slides 13–25). CLIP learns one embedding space for images and text
by contrastive pre-training on web image–caption pairs. In this notebook you will:

1. classify images into labels the model was never trained on (*zero-shot*),
2. see how much the **prompt** matters (`"cat"` vs. `"a photo of a cat"`),
3. search images with text (retrieval), and
4. compute CLIP's **contrastive loss** yourself on a tiny batch.

Runs on CPU in a few minutes; a GPU makes it faster (*Runtime → Change runtime type → T4 GPU*).

In [ ]:
!pip -q install -U transformers

In [ ]:
import torch, requests, io
import matplotlib.pyplot as plt
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
print("device:", device)

Six images from the COCO validation set (the same images many papers use as examples).

In [ ]:
urls = {
    "two cats":       "http://images.cocodataset.org/val2017/000000039769.jpg",
    "living room":    "http://images.cocodataset.org/val2017/000000000139.jpg",
    "bear":           "http://images.cocodataset.org/val2017/000000000285.jpg",
    "bedroom":        "http://images.cocodataset.org/val2017/000000000632.jpg",
    "stop sign":      "http://images.cocodataset.org/val2017/000000000724.jpg",
    "teddy bears":    "http://images.cocodataset.org/val2017/000000000776.jpg",
}
images = {k: Image.open(io.BytesIO(requests.get(u, timeout=30).content)).convert("RGB") for k, u in urls.items()}
fig, axes = plt.subplots(1, 6, figsize=(18, 3))
for ax, (k, im) in zip(axes, images.items()):
    ax.imshow(im); ax.set_title(k); ax.axis("off")
plt.show()

## 1 · Zero-shot classification
CLIP never saw these label lists during training. We turn each label into a sentence, embed it, and pick the label whose text embedding
is closest (cosine similarity) to the image embedding. The softmax over similarities gives a probability per label.

In [ ]:
labels = ["cat", "dog", "bear", "teddy bear", "stop sign", "bed", "sofa", "car"]

@torch.no_grad()
def zero_shot(img, labels, template="a photo of a {}."):
    texts = [template.format(l) for l in labels]
    inputs = processor(text=texts, images=img, return_tensors="pt", padding=True).to(device)
    out = model(**inputs)
    return out.logits_per_image.softmax(dim=-1)[0].cpu()   # logits = scaled cosine similarities

for name, img in images.items():
    p = zero_shot(img, labels)
    top = p.topk(3)
    print(f"{name:12s} ->", ", ".join(f"{labels[i]} {v:.0%}" for v, i in zip(top.values, top.indices)))

## 2 · The prompt matters
The CLIP paper reports that wrapping the label in a sentence such as *"a photo of a {label}."* noticeably improves zero-shot accuracy,
because training captions are sentences, not single words. Compare the confidence of the correct label:

In [ ]:
for name, correct in [("two cats", "cat"), ("living room", "sofa")]:
    print(name)
    for tpl in ["{}", "a photo of a {}.", "a blurry photo of a {}.", "a drawing of a {}."]:
        p = zero_shot(images[name], labels, tpl)
        print(f"   {tpl:26s} P({correct}) = {p[labels.index(correct)]:.3f}")

## 3 · Text → image retrieval
Embed all images once, embed a text query, rank images by cosine similarity. This is how CLIP powers image search and
filters web data (Part 2, slide 24).

In [ ]:
@torch.no_grad()
def embed_images(imgs):
    x = processor(images=imgs, return_tensors="pt").to(device)
    e = model.get_image_features(**x)
    return e / e.norm(dim=-1, keepdim=True)

@torch.no_grad()
def embed_texts(texts):
    x = processor(text=texts, return_tensors="pt", padding=True).to(device)
    e = model.get_text_features(**x)
    return e / e.norm(dim=-1, keepdim=True)

names = list(images)
I = embed_images(list(images.values()))
queries = ["something you can sleep on", "animals that are not real", "a traffic rule", "pets resting together", "a wild animal"]
T = embed_texts(queries)
sim = (T @ I.T).cpu()
plt.figure(figsize=(9, 3.5)); plt.imshow(sim, cmap="Reds")
plt.xticks(range(len(names)), names, rotation=30); plt.yticks(range(len(queries)), queries); plt.colorbar(); plt.title("cosine similarity")
plt.show()
for q, row in zip(queries, sim):
    print(f"{q:30s} -> {names[row.argmax()]}")

Look at *"animals that are not real"*: CLIP may pick the **real** bear instead of the teddy bears. Contrastive training on captions is
good at matching objects and styles, but weak at negation and abstract logic — one motivation for putting an LLM on top (Part 3).

## 4 · CLIP's contrastive loss, by hand
For a batch of N matching (image, text) pairs, CLIP builds an N×N similarity matrix. The diagonal holds the true pairs. The loss is the
average of two cross-entropies: pick the right text for each image (rows) and the right image for each text (columns). This is the
objective on slides 20–21.

In [ ]:
import torch.nn.functional as F
captions = ["two cats lying on a pink couch", "a living room with a TV and plants", "a brown bear in the grass",
            "a bedroom with a bed", "a red stop sign by the road", "teddy bears sitting together"]
with torch.no_grad():
    Ti = embed_texts(captions)
    logit_scale = model.logit_scale.exp()            # learned temperature, about 100
    logits = logit_scale * I @ Ti.T                  # [N images, N texts]
target = torch.arange(len(captions), device=logits.device)
loss = (F.cross_entropy(logits, target) + F.cross_entropy(logits.T, target)) / 2
print("contrastive loss on matched batch:", round(loss.item(), 4))
shuffled = logits[:, torch.randperm(len(captions))]
print("loss if captions were shuffled:   ", round(((F.cross_entropy(shuffled, target) + F.cross_entropy(shuffled.T, target)) / 2).item(), 4))

## Exercises
1. Add a label CLIP gets wrong. Can a better prompt template fix it?
2. Write a query in Chinese. CLIP ViT-B/32 was trained mostly on English text — what happens? (Multilingual CLIP models fix this.)
3. Replace the model with `openai/clip-vit-large-patch14` and compare zero-shot confidence. What does a bigger encoder buy you?
4. In section 4, what happens to the loss if you divide `logit_scale` by 10? Why does the temperature matter?

**Next:** [Notebook 2 — chat with an open multimodal LLM](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture10/lecture10_2_chat_with_an_mllm.ipynb)